# 04 — Model 1: Interaction Logit + AME

Causal model with heterogeneous treatment effect using:
`has_booster * internet_usage`

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from pathlib import Path

df = pd.read_csv(Path("processed/df_analysis.csv"))
df["internet_usage"] = pd.Categorical(
    df["internet_usage"],
    categories=["Low", "Medium", "High", "Extreme"],
    ordered=True,
)
for col in ["tv_product", "mobile_product", "commune"]:
    df[col] = df[col].astype("category")

model_cols = [
    "churned", "has_booster", "age", "tenure",
    "internet_usage", "tv_product", "mobile_product", "commune",
]
df_clean = df[model_cols].dropna().copy()
print(f"Rows for model: {len(df_clean):,}")

In [ ]:
formula_interaction = (
    "churned ~ has_booster * C(internet_usage) + age + tenure + "
    "C(tv_product) + C(mobile_product) + C(commune)"
)
logit_interaction = smf.logit(formula=formula_interaction, data=df_clean).fit(disp=False)
print(logit_interaction.summary())

In [ ]:
def compute_effects_from_model(fitted_model, frame):
    f1 = frame.copy(); f0 = frame.copy()
    f1["has_booster"] = 1; f0["has_booster"] = 0
    ite = fitted_model.predict(f1) - fitted_model.predict(f0)
    temp = frame.assign(ite_churn=ite)
    overall = float(temp["ite_churn"].mean())
    by_tier = temp.groupby("internet_usage", observed=True)["ite_churn"].mean()
    return overall, by_tier, temp

def delta_method_ame(fitted_model, df_subset):
    f1 = df_subset.copy(); f0 = df_subset.copy()
    f1["has_booster"] = 1; f0["has_booster"] = 0
    p1 = fitted_model.predict(f1).to_numpy()
    p0 = fitted_model.predict(f0).to_numpy()
    ame = float(np.mean(p1 - p0))
    exog1, _ = fitted_model._transform_predict_exog(f1)
    exog0, _ = fitted_model._transform_predict_exog(f0)
    exog1 = np.asarray(exog1, dtype=float)
    exog0 = np.asarray(exog0, dtype=float)
    grad = np.mean((p1 * (1 - p1))[:, None] * exog1 - (p0 * (1 - p0))[:, None] * exog0, axis=0)
    var_ame = float(grad @ fitted_model.cov_params() @ grad)
    se = np.sqrt(max(var_ame, 0.0))
    return ame, ame - 1.96 * se, ame + 1.96 * se

In [ ]:
ame_overall, _, temp = compute_effects_from_model(logit_interaction, df_clean)
tier_effects = temp.groupby("internet_usage", observed=True)["ite_churn"].agg(mean="mean", count="count")

# Overall delta CI from get_margeff
mfx = logit_interaction.get_margeff(at="overall", method="dydx")
mfx_row = mfx.summary_frame().loc["has_booster"]
dm_low = float(mfx_row["Conf. Int. Low"])
dm_high = float(mfx_row["Cont. Int. Hi."])

# Tier-level delta CIs
tier_dm = {}
for t in tier_effects.index:
    sub = df_clean[df_clean["internet_usage"].astype(str) == str(t)].copy()
    _, lo, hi = delta_method_ame(logit_interaction, sub)
    tier_dm[str(t)] = (lo, hi)
tier_effects["ci_low_delta"] = [tier_dm[str(t)][0] for t in tier_effects.index]
tier_effects["ci_high_delta"] = [tier_dm[str(t)][1] for t in tier_effects.index]

# Bootstrap CIs
B = 200
rng = np.random.default_rng(42)
boot_overall = []
boot_by_tier = []
for _ in range(B):
    idx = rng.integers(0, len(df_clean), len(df_clean))
    bdf = df_clean.iloc[idx].copy()
    try:
        bfit = smf.logit(formula=formula_interaction, data=bdf).fit(disp=False)
        b_overall, b_tier, _ = compute_effects_from_model(bfit, bdf)
        boot_overall.append(b_overall)
        boot_by_tier.append(b_tier.reindex(tier_effects.index))
    except Exception:
        continue

boot_overall = np.array(boot_overall)
boot_tier = pd.DataFrame(boot_by_tier)
bs_low, bs_high = np.quantile(boot_overall, [0.025, 0.975])
tier_effects["ci_low_boot"] = boot_tier.quantile(0.025).values
tier_effects["ci_high_boot"] = boot_tier.quantile(0.975).values

print(f"Overall AME: {ame_overall*100:+.2f} pp")
print(f"Overall Delta 95% CI: [{dm_low*100:+.2f}, {dm_high*100:+.2f}] pp")
print(f"Overall Bootstrap 95% CI: [{bs_low*100:+.2f}, {bs_high*100:+.2f}] pp")

display(
    tier_effects.reset_index().assign(
        booster_effect_pp=lambda d: d["mean"] * 100,
        ci_low_delta_pp=lambda d: d["ci_low_delta"] * 100,
        ci_high_delta_pp=lambda d: d["ci_high_delta"] * 100,
        ci_low_boot_pp=lambda d: d["ci_low_boot"] * 100,
        ci_high_boot_pp=lambda d: d["ci_high_boot"] * 100,
    )[["internet_usage", "count", "booster_effect_pp", "ci_low_delta_pp", "ci_high_delta_pp", "ci_low_boot_pp", "ci_high_boot_pp"]]
)